# **Preprocessing the CERT Dataset**:

### Imports & Constants:

In [4]:
import config
from scripts.Preprocessing import *
from config import CERT_PATH ,DEFAULT_OUTPUT_DIR
MV = config.MODEL_VERSION

In [5]:
WORK_HOURS = (9, 17)
LONG_URL_THRESHOLD = 90

### Building Layer A: (User, PC, Day) Level Dataset

In [6]:
layer_a_dataset, nunique_frames = build_layer_a(
    cert_path=CERT_PATH,
    work_hours=WORK_HOURS,
    return_nunique_frames=True,
    compute_schedules=True,
    save_schedule_to=os.path.join("processed_datasets", DEFAULT_OUTPUT_DIR, "user_work_hours.parquet")
)

Loading raw CERT logs...
Normalizing CERT logs...
  Normalizing logon.csv...
  file.csv: deferred (will normalize per-chunk)
  Normalizing device.csv...
  email.csv: deferred (will normalize per-chunk)
  http.csv: deferred (will normalize per-chunk)
Deriving per-user work-hour schedules from logon history...
  2474/4000 users have a personal schedule; rest fall back to (9, 17).
Extracting logon features...
Extracting file features (chunked)...
  File chunk 1...
  File chunk 2...
  File chunk 3...
  File chunk 4...
  File chunk 5...
  File chunk 6...
  File chunk 7...
  File chunk 8...
  File chunk 9...
  File chunk 10...
  File chunk 11...
  File chunk 12...
  File chunk 13...
  File chunk 14...
  File chunk 15...
  File chunk 16...
  File chunk 17...
  File chunk 18...
  File chunk 19...
  File chunk 20...
  File chunk 21...
  File chunk 22...
  File chunk 23...
  File chunk 24...
  File chunk 25...
  File chunk 26...
  File chunk 27...
  File chunk 28...
  File chunk 29...
  File chu

In [7]:
layer_a_dataset.head()

,user,pc,day,logon_count,logoff_count,off_hours_logon,logon_late_night_count,logon_hourly_entropy,logon_peak_hour_count,logon_longest_active_run_minutes,...,unique_domains_visited,http_hourly_entropy,http_peak_hour_count,pc_prior_use_count,pc_seen_before,pc_prior_use_ratio,pc_is_primary,distinct_pcs_used_prior,n_pcs_used_today,new_pc_after_stable_history
0,aab0162,pc-6599,2010-01-04,1,1,1,0,0,1,0,...,16,2,25,0,0,0.0,1,0,1,0
1,aab0162,pc-6599,2010-01-05,1,1,1,0,0,1,0,...,14,2,17,1,1,1.0,1,1,1,0
2,aab0162,pc-6599,2010-01-06,1,1,1,0,0,1,0,...,19,2,15,2,1,1.0,1,1,1,0
3,aab0162,pc-6599,2010-01-07,1,1,1,0,0,1,0,...,26,2,17,3,1,1.0,1,1,1,0
4,aab0162,pc-6599,2010-01-08,1,1,1,0,0,1,0,...,10,2,17,4,1,1.0,1,1,1,0


In [8]:
layer_a_path = save_dataset(
    dataset=layer_a_dataset,
    filename=f"ueba_dataset_{MV}a.csv",
    output_dir=DEFAULT_OUTPUT_DIR
)

Dataset successfully saved to: c:\Users\loera\Documents\Insider-Threat-Detection\processed_datasets\ueba_dataset_6\ueba_dataset_6a.csv


### Building Layer B: (User, Day) Model Training Dataset

In [11]:
# Loading the LDAP metadata
ldap_df = load_ldap(CERT_PATH)

Loaded LDAP metadata: 4,000 users, 46 unique roles.


In [12]:
layer_b_dataset = build_layer_b(
    layer_a_df=layer_a_dataset,
    rolling_window=5,
    nunique_frames=nunique_frames,
    ldap_df=ldap_df,
    peer_col=config.PEER_GROUP_KEY,
)

Collapsing Layer A to (user, day) granularity...
  Aggregating per-PC behavioral features...
  Computing cross-channel flags...
  Recomputing true nunique counts from raw event frames...
Adding multi-horizon rolling features (7d/30d sums, 1d-over-30d ratios)...


c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1581: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_1d_over_30d_ratio"] = (df[col] / (daily_avg_30d + 0.5)).clip(0, 50)
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1577: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_7d_sum"] = sum_7d
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1578: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling 

Applying UEBA enhancements (z-scores, rolling deltas, risk flags)...
  Computing per-user causal z-score deviations...
  Computing per-user long-horizon (90d) z-score deviations...
  Computing causal rolling mean deltas...
  Adding cross-channel risk flags...
Applying peer-group enhancements (peer_col='role')...


c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1640: RuntimeWarning: divide by zero encountered in divide
  (df[col].values - loo_mean) / loo_std,
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1640: RuntimeWarning: invalid value encountered in divide
  (df[col].values - loo_mean) / loo_std,
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1640: RuntimeWarning: divide by zero encountered in divide
  (df[col].values - loo_mean) / loo_std,
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1640: RuntimeWarning: invalid value encountered in divide
  (df[col].values - loo_mean) / loo_std,
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1640: RuntimeWarning: divide by zero encountered in divide
  (df[col].values - loo_mean) / loo_std,
c:\Users\loera\Documents\Insider-Threat-Detection\scripts\Preprocessing.py:1640: RuntimeWarning: invalid value encountered in d

Layer B complete — 1,394,010 rows, 407 features.


In [13]:
layer_b_dataset.head()

,user,day,logon_count,logoff_count,off_hours_logon,logon_late_night_count,file_open_count,file_write_count,file_copy_count,file_delete_count,...,http_long_url_count_peer_zscore,unique_domains_visited_peer_zscore,http_late_night_count_peer_zscore,http_hourly_entropy_peer_zscore,http_peak_hour_count_peer_zscore,pcs_used_count_peer_zscore,non_primary_pc_used_flag_peer_zscore,non_primary_pc_http_requests_flag_peer_zscore,non_primary_pc_usb_flag_peer_zscore,non_primary_pc_file_copy_flag_peer_zscore
0,aab0162,2010-01-04,1,1,1,0,0,0,0,0,...,-0.613960,-1.090497,0.000000,0.455265,0.391171,-0.237967,-0.237967,0.0,0.0,0.0
1,aab0162,2010-01-05,1,1,1,0,0,0,0,0,...,-1.329404,-1.637148,-0.094916,0.483123,-0.262860,-0.094916,-0.094916,0.0,0.0,0.0
2,aab0162,2010-01-06,1,1,1,0,0,0,0,0,...,0.899130,-1.064603,0.000000,0.328311,-0.442253,-0.313238,-0.313238,0.0,0.0,0.0
3,aab0162,2010-01-07,1,1,1,0,0,0,0,0,...,0.020504,-0.109382,-0.094916,0.522854,-0.220730,0.000000,0.000000,0.0,0.0,0.0
4,aab0162,2010-01-08,1,1,1,0,0,0,0,0,...,-0.094618,-2.015948,0.000000,0.497096,-0.370468,-0.094916,-0.094916,0.0,0.0,0.0


In [14]:
layer_b_path = save_dataset(
    dataset=layer_b_dataset,
    filename=f"ueba_dataset_{MV}b.csv",
    output_dir=DEFAULT_OUTPUT_DIR
)

Dataset successfully saved to: c:\Users\loera\Documents\Insider-Threat-Detection\processed_datasets\ueba_dataset_6\ueba_dataset_6b.csv


### Creating the Chronological Train/Test Split

In [15]:
# Splitting dataset B
train_df, test_df = chronological_split(df=layer_b_dataset, split_ratio=0.9)

In [ ]:
# Saving training and testing sets
train_path = save_dataset(train_df, f"ueba_dataset_{MV}_train.csv", DEFAULT_OUTPUT_DIR)
test_path = save_dataset(test_df, f"ueba_dataset_{MV}_test_stream.csv", DEFAULT_OUTPUT_DIR)

Dataset successfully saved to: c:\Users\loera\Documents\Insider-Threat-Detection\processed_datasets\ueba_dataset_6\ueba_dataset_6_train.csv
Dataset successfully saved to: c:\Users\loera\Documents\Insider-Threat-Detection\processed_datasets\ueba_dataset_6\ueba_dataset_6_test_stream.csv
Parquet versions saved alongside CSVs.


In [ ]:
# Parquet copies for downstream notebooks (Autoencoder reads parquet)
train_df.to_parquet(train_path.replace(".csv", ".parquet"), index=False)
test_df.to_parquet(test_path.replace(".csv", ".parquet"), index=False)
print(f"Parquet versions saved alongside CSVs.")

###  Utilizing Drill-Down Dataset Lookup

In [17]:
# Example usage
user = "abh0349"
day = "2010-01-27"

In [18]:
drill_down_table = get_drilldown(layer_a_dataset, user, day)

In [19]:
drill_down_table

,user,pc,day,logon_count,logoff_count,off_hours_logon,logon_late_night_count,logon_hourly_entropy,logon_peak_hour_count,logon_longest_active_run_minutes,...,unique_domains_visited,http_hourly_entropy,http_peak_hour_count,pc_prior_use_count,pc_seen_before,pc_prior_use_ratio,pc_is_primary,distinct_pcs_used_prior,n_pcs_used_today,new_pc_after_stable_history
11356,abh0349,pc-1019,2010-01-27,2,1,0,0,1,1,0,...,7,1,3,17,1,0.809524,1,5,2,0
11357,abh0349,pc-1534,2010-01-27,1,1,1,0,0,2,1,...,0,0,0,0,0,0.000000,0,5,2,1
